# Simulation timing profile

This notebook profiles the sample-data simulation loop with optional warmup and repeated runs.

In [1]:
import sys
import time
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd().parent))

from config import get_development_config, setup_dev_directories, get_simulation_params
from src.caching import load_simulation_caches
from src.data_loader import load_electricity_assets, load_hazard_maps
from src.simulation import simulate_asset_damage_recovery_access_breakdown
from src.timing_profiler import (
    SimulationTimingProfiler,
    run_profiled_runs,
    summarize_profiled_runs,
)


In [2]:
config = get_development_config()
setup_dev_directories(config, remove_cache=False)
simulation_params = get_simulation_params(config)
config['simulation_config']['major_timestep'] = 24

hazard_maps = load_hazard_maps(config['hazard_dir'], max_days=None)
gdf_assets = load_electricity_assets(config['electricity_dir'], asset_types=['msls'])
caches = load_simulation_caches(config['interim_dir'], config['hazard_dir'])

print(f'Assets: {len(gdf_assets)}')
print(f'Hazard maps: {len(hazard_maps)}')

Created directories: c:\repos\powerpath\powerpath\data\interim\interim_test_hazard_timesteps, c:\repos\powerpath\powerpath\data\output\output_test_hazard_timesteps
Found 10 hazard map files
Found 1 electricity station files matching types ['msls']
All .shp files in directory: ['ls_stations_test_samples.shp', 'msls_stations_test_samples.shp', 'ms_stations_test_samples.shp']
Files with 'station': ['ls_stations_test_samples.shp', 'msls_stations_test_samples.shp', 'ms_stations_test_samples.shp']
Final matched files: ['msls_stations_test_samples.shp']
Loading electricity assets from msls_stations_test_samples.shp
Loaded 245 msls assets
Combined total: 245 electricity assets
Asset types: {'msls': 245}
Loading simulation caches...
No accessibility cache found at c:\repos\powerpath\powerpath\data\interim\interim_test_hazard_timesteps\accessibility_cache_test_hazard_timesteps.pkl
No hazard extraction cache found at c:\repos\powerpath\powerpath\data\interim\interim_test_hazard_timesteps\hazard_e

In [3]:
def run_profiled_sample(profiler, *, execution_id='timing_profile_run'):
    return simulate_asset_damage_recovery_access_breakdown(
        gdf_assets,
        hazard_maps,
        number_repair_crews=simulation_params['number_repair_crews'],
        repair_crew_assignment_method=simulation_params['repair_crew_assignment_method'],
        flood_threshold=simulation_params['flood_threshold'],
        recovery_parameters=simulation_params['recovery_parameters'],
        root_dir=config['root_dir'],
        verbose=False,
        timestep_output=True,
        execution_id=execution_id,
        config=config,
        major_timestep=config['simulation_config']['major_timestep'],
        accessibility_cache=caches.get('accessibility_cache'),
        hazard_extraction_cache=caches.get('hazard_extraction_cache'),
        overlap_cache=caches.get('overlap_cache'),
        island_cache=caches.get('island_cache'),
        profiler=profiler,
    )


In [4]:
# Warmup + single profiled run
profiled_runs = run_profiled_runs(
    lambda profiler: run_profiled_sample(profiler, execution_id=f'timing_{int(time.time())}'),
    runs=1,
    warmup_runs=0,
)

single_profiler = profiled_runs[0]['profiler']
phase_summary = single_profiler.phase_summary()
timestep_summary = single_profiler.timestep_summary()
timestep_phase_summary = single_profiler.timestep_phase_summary()

print('=== Overall phase totals ===')
print(phase_summary.to_string(index=False))
print('\n=== Per-phase call count / average ===')
print(phase_summary[['phase', 'call_count', 'avg_seconds']].to_string(index=False))
print('\n=== Per-timestep timing summary ===')
print(timestep_summary[['timestep', 'total_seconds']].to_string(index=False))
print('\n=== Per-timestep phase aggregate ===')
print(timestep_phase_summary.to_string(index=False))

Removed 0 edges (adjusted thresholds)
After deduplication: 1000 road segments
Buffered 1000 road segments with 20m buffer
Island distribution: {0: 993, 1: 2, 2: 1, 3: 1, 4: 1, 5: 1, 6: 1}
Identified 16 boundary assets out of 245 total assets.
Cached 16 boundary assets out of 245 total assets.
Cached 7 boundary island geographic features from EV0_ma
Cached asset access rfids for 245 assets.
Extracting hazard values using method: max
Saved hazard extraction cache: 1 entries to c:\repos\powerpath\powerpath\data\interim\interim_test_hazard_timesteps\hazard_extraction_cache_test_hazard_timesteps.pkl
Cached hazard extraction results for map 0 from GHG_timesteps_test0.tif
Cache miss for island_assignment_EV0_ma_0.2_n245_ffb9075134a2efe7ad5cdcebb95eef31, computing islands on the fly...
Loading hazard graph from c:\repos\powerpath\powerpath\data\test_samples\static\output_graph\base_graph_hazard_editted.p
Removed 0 edges (adjusted thresholds)
After deduplication: 1000 road segments
Buffered 100